# IG-CAM — Évaluation Expérimentale Complète
**Plan expérimental pour publication scientifique**

## Méthodes comparées
| Méthode | Type | Référence |
|---------|------|-----------|
| Grad-CAM | Gradient unique + GAP | Selvaraju et al., 2017 |
| Grad-CAM++ | Gradient pixel-wise pondéré | Chattopadhyay et al., 2018 |
| Score-CAM | Sans gradient (perturbation) | Wang et al., 2020 |
| Layer-CAM | Gradient pixel-wise + ReLU | Jiang et al., 2021 |
| Integrated Gradients (input) | IG au niveau pixels | Sundararajan et al., 2017 |
| **IG-CAM (Variante A)** | IG conv + GAP scalaire | **Proposé** |
| **IG-CAM++ (Variante B)** | IG conv + poids pixel-wise | **Proposé** |

## Modèles
VGG-16 · ResNet-50 · ResNet-152 · DenseNet-201 · EfficientNet-B4 (ImageNet pré-entraîné)

## Datasets
ImageNet (2 000 images) · CUB-200-2011 · ISIC 2019 · PASCAL VOC 2007

## Métriques
Deletion AUC ↓ · Insertion AUC ↑ · Pointing Game ↑ · Energy-Based Pointing ↑ · Complétude IG

## 1. Setup & Imports

In [ ]:
import sys, os, time, warnings
sys.path.insert(0, os.path.abspath('..'))
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tvm
import torchvision.transforms as T
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from PIL import Image
from pathlib import Path
from tqdm.notebook import tqdm
from scipy.stats import wilcoxon
from scipy.ndimage import gaussian_filter
from itertools import product

# IG-CAM (notre méthode)
from ig_cam import IGCAM, overlay_cam_on_image

# pytorch-grad-cam
try:
    from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM, LayerCAM
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    GRADCAM_LIB = True
    print('✓ pytorch-grad-cam')
except ImportError:
    GRADCAM_LIB = False
    print('✗ pytorch-grad-cam manquant  →  pip install grad-cam')

# captum (Integrated Gradients input-level)
try:
    from captum.attr import IntegratedGradients as CaptumIG
    CAPTUM = True
    print('✓ captum')
except ImportError:
    CAPTUM = False
    print('✗ captum manquant  →  pip install captum')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
np.random.seed(42); torch.manual_seed(42)
print(f'\nDevice : {DEVICE}  |  PyTorch {torch.__version__}')

## 2. Configuration

In [ ]:
# ─── Chemins datasets ────────────────────────────────────────────────────────
# Adapter ces chemins selon votre configuration locale / Kaggle / cluster
PATHS = {
    'imagenet_val':   None,  # ex: '/data/imagenet/val'
    'imagenet_bbox':  None,  # ex: '/data/imagenet/val_bbox'  (annotations XML ILSVRC)
    'cub200_root':    None,  # ex: '/data/CUB_200_2011'
    'isic_imgs':      None,  # ex: '/data/ISIC2019/ISIC_2019_Training_Input'
    'isic_csv':       None,  # ex: '/data/ISIC2019/ISIC_2019_Training_GroundTruth.csv'
    'voc_root':       None,  # ex: '/data/VOCdevkit/VOC2007'
    'results':        './igcam_exp_results',
    'figures':        './igcam_exp_figures',
}
Path(PATHS['results']).mkdir(exist_ok=True)
Path(PATHS['figures']).mkdir(exist_ok=True)

# ─── Paramètres globaux ───────────────────────────────────────────────────────
N_STEPS_DEFAULT  = 50      # pas d'intégration IG-CAM (défaut)
N_STEPS_AUC      = 100     # pas Insertion/Deletion (Petsiuk et al.)
N_IMAGES_IMAGENET= 2000    # images ImageNet à évaluer
N_IMAGES_CUB     = 5794    # toutes les images test CUB
N_IMAGES_ISIC    = 2500    # images ISIC
N_IMAGES_VOC     = 4952    # images VOC 2007 test
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ─── Datasets actifs ─────────────────────────────────────────────────────────
# Mettre à False les datasets non disponibles pour une exécution partielle
ACTIVE_DATASETS = {
    'imagenet': PATHS['imagenet_val'] is not None,
    'cub200':   PATHS['cub200_root']  is not None,
    'isic':     PATHS['isic_imgs']    is not None,
    'voc':      PATHS['voc_root']     is not None,
}

# ─── Modèles actifs ───────────────────────────────────────────────────────────
ACTIVE_MODELS = ['resnet50', 'vgg16', 'resnet152', 'densenet201', 'efficientnet_b4']

# ─── Méthodes actives ─────────────────────────────────────────────────────────
# Score-CAM est très lent (centaines de forward passes) — désactiver si besoin
ACTIVE_METHODS = ['gradcam', 'gradcam_pp', 'scorecam', 'layercam', 'ig_input', 'igcam_a', 'igcam_b']
SKIP_SCORECAM  = False   # True pour accélérer les tests

METHOD_LABELS = {
    'gradcam':   'Grad-CAM',
    'gradcam_pp':'Grad-CAM++',
    'scorecam':  'Score-CAM',
    'layercam':  'Layer-CAM',
    'ig_input':  'IG (input)',
    'igcam_a':   'IG-CAM (A)',
    'igcam_b':   'IG-CAM++ (B)',
}
METHOD_COLORS = {
    'gradcam':'#1f77b4','gradcam_pp':'#ff7f0e','scorecam':'#2ca02c',
    'layercam':'#9467bd','ig_input':'#8c564b','igcam_a':'#e377c2','igcam_b':'#d62728',
}

print('Configuration chargée.')
for k, v in ACTIVE_DATASETS.items():
    print(f'  Dataset  {k:<14s}: {"✓ actif" if v else "✗ désactivé (chemin non défini)"}')

## 3. Model Zoo

VGG-16 · ResNet-50 · ResNet-152 · DenseNet-201 · EfficientNet-B4 (poids ImageNet torchvision)

In [ ]:
# ─── Couche cible par modèle (plan expérimental §1.2) ────────────────────────
def get_target_layer(model, name):
    """Retourne la couche cible pour Grad-CAM / IG-CAM."""
    if name == 'vgg16':
        return model.features[-1]          # MaxPool après dernière conv
    elif name in ('resnet50', 'resnet152'):
        return model.layer4[-1].conv3      # conv3 du dernier Bottleneck
    elif name == 'densenet201':
        last_block = list(model.features.denseblock4.children())[-1]
        return last_block.conv2            # dernière conv du dernier DenseLayer
    elif name == 'efficientnet_b4':
        return model.features[-1][0]       # premier conv du dernier bloc
    else:
        raise ValueError(f'Modèle inconnu : {name}')


# ─── Chargement des modèles (poids torchvision ImageNet) ─────────────────────
def load_model(name):
    """Charge un modèle pré-entraîné en mode eval."""
    if name == 'vgg16':
        m = tvm.vgg16(weights=tvm.VGG16_Weights.IMAGENET1K_V1)
    elif name == 'resnet50':
        m = tvm.resnet50(weights=tvm.ResNet50_Weights.IMAGENET1K_V1)
    elif name == 'resnet152':
        m = tvm.resnet152(weights=tvm.ResNet152_Weights.IMAGENET1K_V1)
    elif name == 'densenet201':
        m = tvm.densenet201(weights=tvm.DenseNet201_Weights.IMAGENET1K_V1)
    elif name == 'efficientnet_b4':
        m = tvm.efficientnet_b4(weights=tvm.EfficientNet_B4_Weights.IMAGENET1K_V1)
    else:
        raise ValueError(name)
    m.eval().to(DEVICE)
    return m


# ─── Instancier tous les modèles actifs ──────────────────────────────────────
MODELS = {}
TARGET_LAYERS = {}
for mname in ACTIVE_MODELS:
    print(f'Chargement {mname} ...', end=' ')
    MODELS[mname]       = load_model(mname)
    TARGET_LAYERS[mname] = get_target_layer(MODELS[mname], mname)
    n_params = sum(p.numel() for p in MODELS[mname].parameters())
    print(f'✓  {n_params/1e6:.1f}M paramètres  |  couche cible : {TARGET_LAYERS[mname].__class__.__name__}')

## 4. Wrappers XAI (7 méthodes)

In [ ]:
# ─── Prétraitement ────────────────────────────────────────────────────────────
preprocess = T.Compose([T.Resize(256), T.CenterCrop(224),
                        T.ToTensor(), T.Normalize(MEAN, STD)])
viz_tf     = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])


def _norm(cam):
    """Normalise une carte en [0,1]."""    cam = cam.astype(np.float32)
    mn, mx = cam.min(), cam.max()
    return (cam - mn) / (mx - mn + 1e-8)


class XAIExplainer:
    """Wrapper unifié pour les 7 méthodes XAI."""

    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model      = MODELS[model_name]
        self.tl         = TARGET_LAYERS[model_name]

    # ── Grad-CAM ───────────────────────────────────────────────────────────
    def gradcam(self, tensor, cls):
        assert GRADCAM_LIB, 'pytorch-grad-cam requis'
        gc  = GradCAM(model=self.model, target_layers=[self.tl])
        cam = gc(input_tensor=tensor, targets=[ClassifierOutputTarget(cls)])[0]
        return _norm(cam)

    # ── Grad-CAM++ ─────────────────────────────────────────────────────────
    def gradcam_pp(self, tensor, cls):
        assert GRADCAM_LIB
        gc  = GradCAMPlusPlus(model=self.model, target_layers=[self.tl])
        cam = gc(input_tensor=tensor, targets=[ClassifierOutputTarget(cls)])[0]
        return _norm(cam)

    # ── Score-CAM ──────────────────────────────────────────────────────────
    def scorecam(self, tensor, cls):
        assert GRADCAM_LIB
        gc  = ScoreCAM(model=self.model, target_layers=[self.tl])
        cam = gc(input_tensor=tensor, targets=[ClassifierOutputTarget(cls)])[0]
        return _norm(cam)

    # ── Layer-CAM ──────────────────────────────────────────────────────────
    def layercam(self, tensor, cls):
        assert GRADCAM_LIB
        gc  = LayerCAM(model=self.model, target_layers=[self.tl])
        cam = gc(input_tensor=tensor, targets=[ClassifierOutputTarget(cls)])[0]
        return _norm(cam)

    # ── Integrated Gradients (niveau pixels d'entrée) ──────────────────────
    def ig_input(self, tensor, cls):
        assert CAPTUM, 'captum requis'
        ig  = CaptumIG(self.model)
        bl  = torch.zeros_like(tensor).to(DEVICE)
        attr = ig.attribute(tensor.to(DEVICE), baselines=bl,
                            target=cls, n_steps=N_STEPS_DEFAULT)
        saliency = attr.squeeze().abs().sum(dim=0).cpu().numpy()
        return _norm(saliency)

    # ── IG-CAM (A) ─────────────────────────────────────────────────────────
    def igcam_a(self, tensor, cls, n_steps=None):
        n  = n_steps or N_STEPS_DEFAULT
        ig = IGCAM(self.model, self.tl, n_steps=n, variant='A')
        cam = ig.generate(tensor.to(DEVICE), target_class=cls)
        compl = ig.verify_completeness(tensor.to(DEVICE), target_class=cls)
        ig.remove_hooks()
        return _norm(cam), compl

    # ── IG-CAM++ (B) ───────────────────────────────────────────────────────
    def igcam_b(self, tensor, cls, n_steps=None):
        n  = n_steps or N_STEPS_DEFAULT
        ig = IGCAM(self.model, self.tl, n_steps=n, variant='B')
        cam = ig.generate(tensor.to(DEVICE), target_class=cls)
        compl = ig.verify_completeness(tensor.to(DEVICE), target_class=cls)
        ig.remove_hooks()
        return _norm(cam), compl

    def explain(self, method: str, tensor, cls, **kw):
        """Interface unifiée : retourne (cam, compl_dict|None, time_ms)."""
        t0 = time.perf_counter()
        if method == 'gradcam':   cam, c = self.gradcam(tensor, cls),   None
        elif method == 'gradcam_pp': cam, c = self.gradcam_pp(tensor, cls), None
        elif method == 'scorecam':   cam, c = self.scorecam(tensor, cls),   None
        elif method == 'layercam':   cam, c = self.layercam(tensor, cls),   None
        elif method == 'ig_input':   cam, c = self.ig_input(tensor, cls),   None
        elif method == 'igcam_a':    cam, c = self.igcam_a(tensor, cls, **kw)
        elif method == 'igcam_b':    cam, c = self.igcam_b(tensor, cls, **kw)
        else: raise ValueError(method)
        dt = (time.perf_counter() - t0) * 1000
        return cam, c, dt


# Instancier un explainer par modèle
EXPLAINERS = {m: XAIExplainer(m) for m in ACTIVE_MODELS}
print('Explainers instanciés pour :', list(EXPLAINERS.keys()))

## 5. Métriques d'évaluation

Deletion/Insertion (Petsiuk 2018) · Pointing Game (Zhang 2018) · Energy-Based Pointing · Complétude IG

In [ ]:
# ─── Insertion / Deletion (Petsiuk et al., 2018) ─────────────────────────────
def insertion_deletion_auc(model, tensor, cam, cls, num_steps=N_STEPS_AUC):
    """
    Retourne (auc_ins, auc_del, ins_scores, del_scores).
    Deletion  : remplace pixels importants par gris moyen (0.5 normalisé).
    Insertion : révèle depuis image noire.
    """
    img = tensor.squeeze(0).cpu().numpy()  # (3,H,W)
    H, W = img.shape[1], img.shape[2]
    order = np.argsort(cam.flatten())[::-1]
    # baselines
    bl_del = np.full_like(img, 0.5)       # gris moyen (approx pixel moyen normalisé)
    bl_ins = np.zeros_like(img)

    step = max(1, H*W // num_steps)
    ins_s, del_s = [], []

    for s in range(num_steps + 1):
        n = min(s*step, H*W)
        mask = np.zeros(H*W, dtype=bool); mask[order[:n]] = True; mask = mask.reshape(H,W)
        ins_img = bl_ins.copy(); del_img = img.copy()
        for c in range(3):
            ins_img[c][mask] = img[c][mask]
            del_img[c][mask] = bl_del[c][mask]
        with torch.no_grad():
            def sc(arr):
                t = torch.tensor(arr).unsqueeze(0).float().to(DEVICE)
                return F.softmax(model(t), dim=1)[0, cls].item()
            ins_s.append(sc(ins_img)); del_s.append(sc(del_img))

    x = np.linspace(0, 1, len(ins_s))
    return np.trapz(ins_s, x), np.trapz(del_s, x), ins_s, del_s


# ─── Pointing Game (Zhang et al., 2018) ───────────────────────────────────────
def pointing_game(cam, bbox):
    """
    cam  : ndarray (H, W) normalisé.
    bbox : (x1, y1, x2, y2) en pixels, référencés sur (224, 224).
    Retourne 1 (hit) ou 0 (miss).
    """
    H, W = cam.shape
    y_max, x_max = np.unravel_index(cam.argmax(), cam.shape)
    x1, y1, x2, y2 = bbox
    # Clamp bbox to image
    x1 = max(0, min(x1, W-1)); x2 = max(0, min(x2, W-1))
    y1 = max(0, min(y1, H-1)); y2 = max(0, min(y2, H-1))
    return int(x1 <= x_max <= x2 and y1 <= y_max <= y2)


# ─── Energy-Based Pointing Game ───────────────────────────────────────────────
def energy_pointing_game(cam, bbox):
    """
    Proportion de l'énergie totale de la saillance dans la bbox.
    Plus informatif que le Pointing Game binaire.
    """
    H, W = cam.shape
    x1, y1, x2, y2 = [max(0, int(v)) for v in bbox]
    x2 = min(x2, W); y2 = min(y2, H)
    inside_energy = cam[y1:y2, x1:x2].sum()
    total_energy  = cam.sum() + 1e-10
    return float(inside_energy / total_energy)


# ─── Complétude IG-CAM ────────────────────────────────────────────────────────
def completeness_error(compl_dict):
    """Extrait l'erreur relative (%) depuis le dict de verify_completeness."""    return compl_dict['relative_error'] * 100 if compl_dict else float('nan')


print('Métriques définies : Insertion/Deletion, Pointing Game, Energy Pointing, Complétude.')

## 6. Chargement des datasets

ImageNet · CUB-200-2011 · ISIC 2019 · PASCAL VOC 2007

In [ ]:
# ─── Prétraitement standard ───────────────────────────────────────────────────
preprocess_224 = T.Compose([
    T.Resize(256), T.CenterCrop(224), T.ToTensor(), T.Normalize(MEAN, STD)
])
viz_224 = T.Compose([T.Resize(256), T.CenterCrop(224), T.ToTensor()])


def load_pil(path):
    return Image.open(path).convert('RGB')

def to_tensor(pil):
    return preprocess_224(pil).unsqueeze(0).to(DEVICE)

def to_viz(pil):
    return viz_224(pil).permute(1,2,0).numpy()

def predict_top1(model, tensor):
    with torch.no_grad():
        probs = F.softmax(model(tensor), dim=1)[0]
    cls = probs.argmax().item()
    return cls, float(probs[cls])


# ─── ImageNet (ILSVRC 2012 val) ───────────────────────────────────────────────
def load_imagenet(root, bbox_root=None, n_images=2000, seed=42):
    """
    Charge n_images depuis la validation ImageNet.
    Retourne une liste de dicts {image_id, path, tensor, viz, true_class, bbox}.
    bbox est None si bbox_root n'est pas fourni.
    """
    import xml.etree.ElementTree as ET
    root = Path(root)
    # ImageNet val : dossiers de classes ou images plates
    all_imgs = sorted(root.rglob('*.JPEG')) + sorted(root.rglob('*.jpg'))
    rng = np.random.default_rng(seed)
    if len(all_imgs) > n_images:
        all_imgs = [all_imgs[i] for i in rng.choice(len(all_imgs), n_images, replace=False)]

    items = []
    for p in tqdm(all_imgs, desc='ImageNet'):
        try:
            pil  = load_pil(str(p))
            t    = to_tensor(pil)
            viz  = to_viz(pil)
            cls, conf = predict_top1(MODELS['resnet50'], t)  # use ResNet50 as predictor

            bbox = None
            if bbox_root:
                xml_path = Path(bbox_root) / (p.stem + '.xml')
                if xml_path.exists():
                    tree = ET.parse(xml_path)
                    obj  = tree.find('.//bndbox')
                    if obj is not None:
                        orig_w = int(tree.find('.//width').text)
                        orig_h = int(tree.find('.//height').text)
                        # Scale bbox to 224×224
                        scale_x = 224 / orig_w; scale_y = 224 / orig_h
                        bbox = (
                            int(int(obj.find('xmin').text) * scale_x),
                            int(int(obj.find('ymin').text) * scale_y),
                            int(int(obj.find('xmax').text) * scale_x),
                            int(int(obj.find('ymax').text) * scale_y),
                        )
            items.append({'image_id':p.stem,'path':str(p),'tensor':t,
                          'viz':viz,'true_class':cls,'bbox':bbox,'dataset':'imagenet'})
        except: continue
    print(f'ImageNet : {len(items)} images chargées.')
    return items


# ─── CUB-200-2011 ─────────────────────────────────────────────────────────────
def load_cub200(root, split='test'):
    """
    Charge les images CUB-200-2011 avec bboxes.
    Structure attendue : root/images/, root/bounding_boxes.txt, root/train_test_split.txt
    """
    root = Path(root)
    # Lire le split (1=train, 0=test)
    split_flag = 0 if split == 'test' else 1
    splits  = pd.read_csv(root/'train_test_split.txt',    sep=' ', names=['id','is_train'])
    imgs    = pd.read_csv(root/'images.txt',              sep=' ', names=['id','path'])
    bboxes  = pd.read_csv(root/'bounding_boxes.txt',      sep=' ', names=['id','x','y','w','h'])
    labels  = pd.read_csv(root/'image_class_labels.txt',  sep=' ', names=['id','class'])

    df = splits[splits['is_train']==split_flag]         .merge(imgs,   on='id').merge(bboxes, on='id').merge(labels, on='id')

    items = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc='CUB-200'):
        try:
            img_path = root / 'images' / row['path']
            pil = load_pil(str(img_path))
            # Scale bbox original → 224×224
            orig_w, orig_h = pil.size
            scale_x = 224/orig_w; scale_y = 224/orig_h
            bbox = (int(row['x']*scale_x), int(row['y']*scale_y),
                    int((row['x']+row['w'])*scale_x), int((row['y']+row['h'])*scale_y))
            t   = to_tensor(pil); viz = to_viz(pil)
            cls, conf = predict_top1(MODELS['resnet50'], t)
            items.append({'image_id':Path(row['path']).stem,'path':str(img_path),
                          'tensor':t,'viz':viz,'true_class':row['class']-1,
                          'bbox':bbox,'dataset':'cub200'})
        except: continue
    print(f'CUB-200 : {len(items)} images chargées.')
    return items


# ─── ISIC 2019 ────────────────────────────────────────────────────────────────
def load_isic(img_dir, csv_path=None, n_images=2500, seed=42):
    """Charge des images ISIC 2019 (sans bbox — Pointing Game non applicable)."""    img_dir = Path(img_dir)
    all_imgs = sorted(img_dir.glob('*.jpg'))
    rng = np.random.default_rng(seed)
    if len(all_imgs) > n_images:
        all_imgs = [all_imgs[i] for i in rng.choice(len(all_imgs), n_images, replace=False)]

    ISIC_CLASSES = ['MEL','NV','BCC','AK','BKL','DF','VASC','SCC']
    gt_map = {}
    if csv_path and Path(csv_path).exists():
        df_csv = pd.read_csv(csv_path)
        label_cols = [c for c in df_csv.columns if c not in ('image','UNK')]
        df_csv['label'] = df_csv[label_cols].values.argmax(axis=1)
        gt_map = dict(zip(df_csv['image'], df_csv['label']))

    items = []
    for p in tqdm(all_imgs, desc='ISIC'):
        try:
            pil = load_pil(str(p)); t = to_tensor(pil); viz = to_viz(pil)
            cls, conf = predict_top1(MODELS['resnet50'], t)
            true_cls  = gt_map.get(p.stem, None)
            items.append({'image_id':p.stem,'path':str(p),'tensor':t,'viz':viz,
                          'true_class':true_cls,'bbox':None,'dataset':'isic'})
        except: continue
    print(f'ISIC 2019 : {len(items)} images chargées.')
    return items


# ─── PASCAL VOC 2007 ──────────────────────────────────────────────────────────
def load_voc2007(root):
    """Charge PASCAL VOC 2007 test set avec bboxes (première bbox de la première annotation)."""    import xml.etree.ElementTree as ET
    root = Path(root)
    ann_dir = root/'Annotations'
    img_dir = root/'JPEGImages'
    split_file = root/'ImageSets'/'Main'/'test.txt'

    with open(split_file) as f:
        img_ids = [l.strip() for l in f if l.strip()]

    items = []
    for img_id in tqdm(img_ids, desc='VOC 2007'):
        try:
            ann  = ET.parse(ann_dir / f'{img_id}.xml')
            obj  = ann.find('.//bndbox')
            if obj is None: continue
            orig_w = int(ann.find('.//width').text)
            orig_h = int(ann.find('.//height').text)
            scale_x = 224/orig_w; scale_y = 224/orig_h
            bbox = (int(int(obj.find('xmin').text)*scale_x),
                    int(int(obj.find('ymin').text)*scale_y),
                    int(int(obj.find('xmax').text)*scale_x),
                    int(int(obj.find('ymax').text)*scale_y))

            pil = load_pil(str(img_dir/f'{img_id}.jpg'))
            t   = to_tensor(pil); viz = to_viz(pil)
            cls, conf = predict_top1(MODELS['resnet50'], t)
            items.append({'image_id':img_id,'path':str(img_dir/f'{img_id}.jpg'),
                          'tensor':t,'viz':viz,'true_class':None,
                          'bbox':bbox,'dataset':'voc2007'})
        except: continue
    print(f'VOC 2007 : {len(items)} images chargées.')
    return items


# ─── Charger les datasets actifs ─────────────────────────────────────────────
DATASETS = {}
if ACTIVE_DATASETS['imagenet']:
    DATASETS['imagenet'] = load_imagenet(PATHS['imagenet_val'], PATHS['imagenet_bbox'], N_IMAGES_IMAGENET)
if ACTIVE_DATASETS['cub200']:
    DATASETS['cub200']   = load_cub200(PATHS['cub200_root'])
if ACTIVE_DATASETS['isic']:
    DATASETS['isic']     = load_isic(PATHS['isic_imgs'], PATHS['isic_csv'], N_IMAGES_ISIC)
if ACTIVE_DATASETS['voc']:
    DATASETS['voc']      = load_voc2007(PATHS['voc_root'])

print(f'\nDatasets chargés : {list(DATASETS.keys())}')
for k, v in DATASETS.items(): print(f'  {k:<12s}: {len(v)} images')

## 7. Boucle d'évaluation principale

`datasets × modèles × méthodes` → résultats en CSV

In [ ]:
def evaluate_image(img_info, model_name, method, n_steps=None):
    """
    Évalue une méthode XAI sur une image pour un modèle donné.
    Retourne un dict de métriques.
    """
    tensor    = img_info['tensor']
    bbox      = img_info['bbox']
    dataset   = img_info['dataset']
    explainer = EXPLAINERS[model_name]
    model     = MODELS[model_name]

    # Prédiction du modèle courant (pas forcément ResNet-50)
    cls, conf = predict_top1(model, tensor)

    # Générer la carte XAI
    if method in ('igcam_a', 'igcam_b') and n_steps:
        cam, compl, dt = explainer.explain(method, tensor, cls, n_steps=n_steps)
    else:
        cam, compl, dt = explainer.explain(method, tensor, cls)

    # Métriques quantitatives
    auc_ins, auc_del, _, _ = insertion_deletion_auc(model, tensor, cam, cls)

    pg  = pointing_game(cam, bbox)        if bbox is not None else float('nan')
    epg = energy_pointing_game(cam, bbox) if bbox is not None else float('nan')

    return {
        'image_id'    : img_info['image_id'],
        'dataset'     : dataset,
        'model'       : model_name,
        'method'      : method,
        'pred_class'  : cls,
        'pred_conf'   : conf,
        'auc_insertion': auc_ins,
        'auc_deletion' : auc_del,
        'faithfulness' : auc_ins - auc_del,
        'pointing_game': pg,
        'energy_pointing': epg,
        'completeness_err_pct': completeness_error(compl),
        'time_ms'      : dt,
        'cam'          : cam,   # stocké temporairement (retiré avant CSV)
    }

In [ ]:
# ─── Boucle principale ───────────────────────────────────────────────────────
# Modèles × Datasets × Méthodes
# Résultats sauvegardés en CSV par dataset pour permettre des reprises

all_results = []

active_methods = [m for m in ACTIVE_METHODS
                  if not (m == 'scorecam' and SKIP_SCORECAM)]

combos = [(ds, mname) for ds in DATASETS for mname in ACTIVE_MODELS]
print(f'Plan : {len(combos)} combinaisons dataset×modèle  ×  {len(active_methods)} méthodes')

for (ds_name, model_name) in combos:
    images = DATASETS[ds_name]
    print(f'\n▶  {ds_name:<12s}  ×  {model_name}  ({len(images)} images × {len(active_methods)} méthodes)')
    rows = []

    for img_info in tqdm(images, desc=f'{ds_name}/{model_name}', leave=False):
        for method in active_methods:
            try:
                row = evaluate_image(img_info, model_name, method)
                row.pop('cam', None)   # ne pas stocker les arrays dans le CSV
                rows.append(row)
            except Exception as e:
                rows.append({'image_id':img_info['image_id'],'dataset':ds_name,
                             'model':model_name,'method':method,'error':str(e)})

    chunk = pd.DataFrame(rows)
    csv_path = Path(PATHS['results']) / f'results_{ds_name}_{model_name}.csv'
    chunk.to_csv(csv_path, index=False)
    all_results.append(chunk)
    print(f'   → sauvegardé : {csv_path}')

# Consolider
if all_results:
    results_df = pd.concat(all_results, ignore_index=True)
    results_df.to_csv(Path(PATHS['results'])/'results_all.csv', index=False)
    print(f'\n✓ Résultats consolidés : {len(results_df)} lignes')
    print(results_df.groupby(['dataset','model','method'])[
        ['auc_insertion','auc_deletion','pointing_game']].mean().round(4).to_string())
else:
    print('Aucun dataset actif. Configurer PATHS et relancer.')
    results_df = pd.DataFrame()

## 8. Études d'ablation

### 8.1 Effet du nombre de pas N (Tableau 6)

In [ ]:
# ─── 8.1 Ablation : effet du nombre de pas N ─────────────────────────────────
# Protocole : IG-CAM++ (B) sur ImageNet avec ResNet-50
# Métriques : Deletion AUC, erreur de complétude, temps moyen

N_VALUES   = [5, 10, 20, 50, 100, 200]
N_SAMPLE   = 200  # images pour l'ablation
ABLATION_MODEL = 'resnet50'

if 'imagenet' in DATASETS:
    rng    = np.random.default_rng(42)
    sample = [DATASETS['imagenet'][i]
              for i in rng.choice(len(DATASETS['imagenet']), min(N_SAMPLE, len(DATASETS['imagenet'])), replace=False)]

    ablation_n_rows = []
    for N in N_VALUES:
        print(f'N = {N:3d} ...', end=' ')
        del_aucs, compl_errs, times = [], [], []
        for img_info in tqdm(sample, desc=f'N={N}', leave=False):
            cls, _ = predict_top1(MODELS[ABLATION_MODEL], img_info['tensor'])
            row    = evaluate_image(img_info, ABLATION_MODEL, 'igcam_b', n_steps=N)
            del_aucs.append(row['auc_deletion'])
            compl_errs.append(row['completeness_err_pct'])
            times.append(row['time_ms'])
        ablation_n_rows.append({
            'N':N,
            'deletion_auc':  np.mean(del_aucs),
            'completeness_err_pct': np.mean(compl_errs),
            'time_ms': np.mean(times),
        })
        print(f'Del AUC={np.mean(del_aucs):.4f}  Compl={np.mean(compl_errs):.2f}%  t={np.mean(times):.1f}ms')

    ablation_n_df = pd.DataFrame(ablation_n_rows)
    ablation_n_df.to_csv(Path(PATHS['results'])/'ablation_N.csv', index=False)
    print('\nTableau 6 — Effet de N (IG-CAM++ / ResNet-50 / ImageNet):')
    print(ablation_n_df.to_string(index=False))
else:
    print('ImageNet non disponible — ablation N ignorée.')
    ablation_n_df = pd.DataFrame({'N':N_VALUES,'deletion_auc':np.nan,
                                  'completeness_err_pct':np.nan,'time_ms':np.nan})

### 8.2 Effet du choix de baseline (Tableau 7)

In [ ]:
# ─── 8.2 Ablation : effet du type de baseline ─────────────────────────────────
# Protocole : IG-CAM++ (B) sur CUB-200 avec ResNet-50 (N=50)

BASELINES_DEF = {
    'black':          lambda t: torch.zeros_like(t),
    'gaussian_noise': lambda t: torch.randn_like(t) * 0.5,
    'blurred':        lambda t: torch.tensor(
                          np.stack([gaussian_filter(t.squeeze(0).cpu().numpy()[c], sigma=50)
                                    for c in range(3)])).unsqueeze(0).float().to(DEVICE),
    'dataset_mean':   lambda t: torch.tensor(MEAN).view(3,1,1).expand_as(t.squeeze(0))
                          .unsqueeze(0).to(DEVICE),
}

N_SAMPLE_BL = 200
ABLATION_BL_MODEL = 'resnet50'

def igcam_b_custom_baseline(model, target_layer, tensor, cls, baseline_tensor, n_steps=50):
    """IG-CAM++ avec baseline personnalisée."""    ig = IGCAM(model, target_layer, n_steps=n_steps, variant='B')
    cam   = ig.generate(tensor, target_class=cls, baseline=baseline_tensor)
    compl = ig.verify_completeness(tensor, target_class=cls, baseline=baseline_tensor)
    ig.remove_hooks()
    return _norm(cam), compl

ds_bl = 'cub200' if 'cub200' in DATASETS else ('imagenet' if 'imagenet' in DATASETS else None)

if ds_bl:
    rng    = np.random.default_rng(42)
    sample = [DATASETS[ds_bl][i]
              for i in rng.choice(len(DATASETS[ds_bl]), min(N_SAMPLE_BL, len(DATASETS[ds_bl])), replace=False)]

    abl_bl_rows = []
    for bl_name, bl_fn in BASELINES_DEF.items():
        del_aucs, compl_errs = [], []
        print(f'Baseline: {bl_name} ...', end=' ')
        for img_info in tqdm(sample, desc=bl_name, leave=False):
            cls, _ = predict_top1(MODELS[ABLATION_BL_MODEL], img_info['tensor'])
            bl_t   = bl_fn(img_info['tensor'])
            cam, compl = igcam_b_custom_baseline(
                MODELS[ABLATION_BL_MODEL], TARGET_LAYERS[ABLATION_BL_MODEL],
                img_info['tensor'].to(DEVICE), cls, bl_t)
            auc_ins, auc_del, _, _ = insertion_deletion_auc(
                MODELS[ABLATION_BL_MODEL], img_info['tensor'], cam, cls)
            del_aucs.append(auc_del); compl_errs.append(compl['relative_error']*100)
        abl_bl_rows.append({'baseline':bl_name,'deletion_auc':np.mean(del_aucs),
                            'completeness_err_pct':np.mean(compl_errs)})
        print(f'Del AUC={np.mean(del_aucs):.4f}  Compl={np.mean(compl_errs):.2f}%')

    abl_bl_df = pd.DataFrame(abl_bl_rows)
    abl_bl_df.to_csv(Path(PATHS['results'])/'ablation_baseline.csv', index=False)
    print('\nTableau 7 — Effet du baseline :')
    print(abl_bl_df.to_string(index=False))
else:
    print('Aucun dataset disponible pour l\'ablation baseline.')
    abl_bl_df = pd.DataFrame()

## 9. Tests statistiques — Wilcoxon + Bonferroni

In [ ]:
# ─── 9. Tests statistiques ────────────────────────────────────────────────────
# Wilcoxon signé (paired, non-paramétrique) + correction de Bonferroni

BASELINES_STAT = ['gradcam','gradcam_pp','scorecam','layercam','ig_input']
TESTED_METHODS = ['igcam_a','igcam_b']
METRIC         = 'auc_deletion'   # tester aussi 'auc_insertion'

def run_wilcoxon_tests(df, metric=METRIC, dataset=None, model=None):
    """
    Pour chaque (IG-CAM variant) vs (baseline), applique le test de Wilcoxon
    sur les valeurs par image (paired).
    Retourne un DataFrame avec p-values et verdict Bonferroni.
    """
    sub = df.copy()
    if dataset: sub = sub[sub.dataset == dataset]
    if model:   sub = sub[sub.model   == model]

    # Pivoter pour avoir une colonne par méthode, une ligne par image
    try:
        pivot = sub.pivot_table(index='image_id', columns='method', values=metric, aggfunc='mean')
    except Exception as e:
        print(f'Erreur pivot : {e}'); return pd.DataFrame()

    n_comparisons = len(BASELINES_STAT) * len(TESTED_METHODS)
    alpha_bonf    = 0.05 / n_comparisons

    rows = []
    for igm in TESTED_METHODS:
        for bsm in BASELINES_STAT:
            if igm not in pivot.columns or bsm not in pivot.columns:
                continue
            valid = pivot[[igm, bsm]].dropna()
            if len(valid) < 20:
                continue
            # Direction : Deletion AUC → on veut IG-CAM < baseline → alternative='less'
            alt = 'less' if metric == 'auc_deletion' else 'greater'
            stat, pval = wilcoxon(valid[igm], valid[bsm], alternative=alt)
            rows.append({
                'method_A': METHOD_LABELS.get(igm, igm),
                'method_B': METHOD_LABELS.get(bsm, bsm),
                'n_pairs' : len(valid),
                'W_stat'  : round(stat, 2),
                'p_value' : pval,
                'p_bonf'  : min(pval * n_comparisons, 1.0),
                'significant': pval < alpha_bonf,
                'mean_A'  : valid[igm].mean(),
                'mean_B'  : valid[bsm].mean(),
                'delta'   : valid[igm].mean() - valid[bsm].mean(),
            })
    return pd.DataFrame(rows)


if not results_df.empty:
    print(f'=== Tests de Wilcoxon — {METRIC} ===')
    print(f'Correction Bonferroni : α/{len(BASELINES_STAT)*len(TESTED_METHODS)} = '
          f'{0.05/(len(BASELINES_STAT)*len(TESTED_METHODS)):.4f}\n')

    stat_df = run_wilcoxon_tests(results_df, metric=METRIC)
    if not stat_df.empty:
        print(stat_df[['method_A','method_B','n_pairs','p_value','p_bonf',
                        'significant','mean_A','mean_B','delta']].to_string(index=False))
        stat_df.to_csv(Path(PATHS['results'])/'wilcoxon_tests.csv', index=False)
else:
    print('Pas de résultats — tests statistiques ignorés.')
    stat_df = pd.DataFrame()

## 10. Figures du papier

### Figure 1 — Comparaison qualitative

In [ ]:
# ─── Figure 1 — Comparaison qualitative (grille 7 méthodes × 5 images) ───────
# Sélectionner manuellement 5 images représentatives
# ou utiliser les images du premier dataset actif

def figure1_qualitative_grid(images_5, model_name='resnet50',
                              save_path=None):
    """
    Construit une grille 5 lignes × 8 colonnes :
    [Original | Grad-CAM | Grad-CAM++ | Score-CAM | Layer-CAM | IG (input) | IG-CAM A | IG-CAM++ B]
    """
    methods_ordered = ['gradcam','gradcam_pp','scorecam','layercam',
                       'ig_input','igcam_a','igcam_b']
    col_labels = ['Original'] + [METHOD_LABELS[m] for m in methods_ordered]
    n_cols = len(col_labels)
    n_rows = len(images_5)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.2*n_cols, 3.2*n_rows))
    if n_rows == 1: axes = axes[np.newaxis,:]

    for ci, label in enumerate(col_labels):
        axes[0, ci].set_title(label, fontsize=9, fontweight='bold',
                               color=METHOD_COLORS.get(methods_ordered[ci-1] if ci>0 else '', 'black'))

    for ri, img_info in enumerate(images_5):
        tensor = img_info['tensor']
        viz    = img_info['viz']
        cls, _ = predict_top1(MODELS[model_name], tensor)

        # Original
        axes[ri, 0].imshow(viz)
        axes[ri, 0].set_ylabel(f"{img_info.get('image_id','')[:14]}\n{img_info['dataset']}",
                               fontsize=7.5, rotation=0, labelpad=90, va='center')

        # 7 méthodes
        for ci, mname in enumerate(methods_ordered, start=1):
            try:
                cam, _, _ = EXPLAINERS[model_name].explain(mname, tensor, cls)
                overlay   = overlay_cam_on_image(viz, cam, alpha=0.5)
                axes[ri, ci].imshow(overlay)
            except Exception as e:
                axes[ri, ci].text(0.5, 0.5, f'N/A\n{str(e)[:20]}',
                                  ha='center', va='center', transform=axes[ri,ci].transAxes, fontsize=7)

    for ax in axes.flat: ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle('Figure 1 — Comparaison qualitative des méthodes XAI', fontsize=13, y=1.01)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Choisir 5 images du premier dataset actif
first_ds = list(DATASETS.keys())[0] if DATASETS else None
if first_ds:
    rng = np.random.default_rng(0)
    sample5 = [DATASETS[first_ds][i] for i in rng.choice(len(DATASETS[first_ds]), min(5, len(DATASETS[first_ds])), replace=False)]
    figure1_qualitative_grid(sample5, save_path=Path(PATHS['figures'])/'fig1_qualitative_grid.png')
else:
    print('Aucun dataset actif — Figure 1 ignorée.')

### Figure 2 — Courbes Insertion / Deletion

In [ ]:
# ─── Figure 2 — Courbes Deletion / Insertion ─────────────────────────────────
# Sur ImageNet avec ResNet-50 (ou premier dataset actif)

def figure2_curves(images_batch, model_name='resnet50',
                   n_images=50, save_path=None):
    """
    Calcule et trace les courbes moyennes Deletion et Insertion
    pour chaque méthode sur n_images.
    """
    rng     = np.random.default_rng(42)
    sample  = [images_batch[i] for i in rng.choice(len(images_batch), min(n_images, len(images_batch)), replace=False)]
    methods = [m for m in ACTIVE_METHODS if not (m=='scorecam' and SKIP_SCORECAM)]

    all_ins = {m: [] for m in methods}
    all_del = {m: [] for m in methods}

    for img_info in tqdm(sample, desc='Figure 2'):
        cls, _ = predict_top1(MODELS[model_name], img_info['tensor'])
        for mname in methods:
            try:
                cam, _, _ = EXPLAINERS[model_name].explain(mname, img_info['tensor'], cls)
                _, _, ins_s, del_s = insertion_deletion_auc(
                    MODELS[model_name], img_info['tensor'], cam, cls)
                all_ins[mname].append(ins_s); all_del[mname].append(del_s)
            except: pass

    x = np.linspace(0, 1, N_STEPS_AUC + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    for mname in methods:
        if not all_ins[mname]: continue
        ins_mean = np.mean(all_ins[mname], axis=0)
        del_mean = np.mean(all_del[mname], axis=0)
        col   = METHOD_COLORS[mname]
        label = METHOD_LABELS[mname]
        auc_i = np.trapz(ins_mean, x)
        auc_d = np.trapz(del_mean, x)
        ax1.plot(x, ins_mean, color=col, lw=2, label=f'{label} ({auc_i:.3f})')
        ax2.plot(x, del_mean, color=col, lw=2, label=f'{label} ({auc_d:.3f})')

    for ax, title in [(ax1,'Insertion  (AUC ↑ = meilleur)'),(ax2,'Deletion  (AUC ↓ = meilleur)')]:
        ax.set_xlabel('Fraction de pixels insérés/supprimés', fontsize=11)
        ax.set_ylabel('Score du modèle', fontsize=11)
        ax.set_title(title, fontsize=12)
        ax.legend(fontsize=8.5); ax.grid(alpha=0.3)

    fig.suptitle(f'Figure 2 — Courbes Insertion / Deletion — {model_name} / {first_ds}', fontsize=13)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


if first_ds:
    figure2_curves(DATASETS[first_ds],
                   save_path=Path(PATHS['figures'])/'fig2_insertion_deletion.png')

### Figure 3 — Effet de N sur qualité et temps

In [ ]:
# ─── Figure 3 — Effet de N sur la qualité et le temps ────────────────────────
if not ablation_n_df.empty and not ablation_n_df['deletion_auc'].isna().all():
    fig, ax1 = plt.subplots(figsize=(9, 5))
    ax2 = ax1.twinx()

    color_del  = '#d62728'
    color_time = '#1f77b4'

    l1 = ax1.plot(ablation_n_df['N'], ablation_n_df['deletion_auc'],
                  'o-', color=color_del, lw=2.5, ms=8, label='Deletion AUC ↓')
    l2 = ax2.plot(ablation_n_df['N'], ablation_n_df['time_ms'],
                  's--', color=color_time, lw=2, ms=7, label='Temps (ms/image)')

    # Ligne Grad-CAM de référence
    if not results_df.empty:
        gc_del = results_df[(results_df.method=='gradcam') &
                            (results_df.model==ABLATION_MODEL)]['auc_deletion'].mean()
        ax1.axhline(gc_del, color='gray', ls=':', lw=1.5, label=f'Grad-CAM AUC = {gc_del:.3f}')

    ax1.set_xlabel('Nombre de pas N', fontsize=12)
    ax1.set_ylabel('Deletion AUC ↓', color=color_del, fontsize=12)
    ax2.set_ylabel('Temps (ms / image)', color=color_time, fontsize=12)
    ax1.tick_params(axis='y', labelcolor=color_del)
    ax2.tick_params(axis='y', labelcolor=color_time)

    lines = l1 + l2
    if not results_df.empty: lines += ax1.get_lines()[1:]
    ax1.legend(lines, [l.get_label() for l in lines], fontsize=10, loc='upper right')
    ax1.set_title('Figure 3 — Effet de N sur Deletion AUC et temps de calcul
(IG-CAM++ B / ResNet-50 / ImageNet)', fontsize=12)
    ax1.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(PATHS['figures'])/'fig3_ablation_N.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Données ablation N non disponibles — Figure 3 ignorée.')

### Figure 4 — Distribution de l'erreur de complétude

In [ ]:
# ─── Figure 4 — Histogramme de l'erreur de complétude ────────────────────────
if not results_df.empty:
    compl_data = results_df[results_df.method.isin(['igcam_a','igcam_b'])
                            & results_df['completeness_err_pct'].notna()]

    fig, ax = plt.subplots(figsize=(9, 5))
    for mname, col in [('igcam_a', METHOD_COLORS['igcam_a']),
                       ('igcam_b', METHOD_COLORS['igcam_b'])]:
        sub = compl_data[compl_data.method==mname]['completeness_err_pct'].dropna()
        if sub.empty: continue
        pct_under3 = (sub < 3).mean() * 100
        ax.hist(sub, bins=40, alpha=0.6, color=col, label=f'{METHOD_LABELS[mname]}  ({pct_under3:.1f}% < 3%)', density=True)

    ax.axvline(3,  color='green', ls='--', lw=1.5, label='3%')
    ax.axvline(10, color='red',   ls='--', lw=1.5, label='10%')
    ax.set_xlabel('Erreur relative de complétude (%)', fontsize=12)
    ax.set_ylabel('Densité', fontsize=12)
    ax.set_title(f'Figure 4 — Distribution de l\'erreur de complétude  (N={N_STEPS_DEFAULT})', fontsize=12)
    ax.legend(fontsize=10); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(PATHS['figures'])/'fig4_completeness_hist.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Pas de résultats — Figure 4 ignorée.')

### Figure 5 — Effet de la profondeur du réseau

In [ ]:
# ─── Figure 5 — Effet de la profondeur du réseau ─────────────────────────────
# Deletion AUC par méthode × modèle sur ImageNet

MODEL_DEPTH = {'vgg16':16, 'resnet50':50, 'resnet152':152,
               'densenet201':201, 'efficientnet_b4':50}
MODEL_PARAMS = {'vgg16':138, 'resnet50':25.6, 'resnet152':60.2,
                'densenet201':20.0, 'efficientnet_b4':19.3}

if not results_df.empty and 'imagenet' in results_df['dataset'].values:
    sub = results_df[results_df.dataset == 'imagenet']
    pivot_del = sub.groupby(['model','method'])['auc_deletion'].mean().unstack('method')

    models_sorted = sorted(ACTIVE_MODELS, key=lambda m: MODEL_DEPTH.get(m, 0))
    methods_plot  = [m for m in ACTIVE_METHODS if m in pivot_del.columns and not (m=='scorecam' and SKIP_SCORECAM)]

    x = np.arange(len(models_sorted)); width = 0.8 / len(methods_plot)

    fig, ax = plt.subplots(figsize=(13, 6))
    for i, mname in enumerate(methods_plot):
        vals = [pivot_del.loc[m, mname] if m in pivot_del.index else np.nan
                for m in models_sorted]
        offset = (i - len(methods_plot)/2 + 0.5) * width
        ax.bar(x + offset, vals, width, label=METHOD_LABELS[mname],
               color=METHOD_COLORS[mname], alpha=0.85, edgecolor='k', lw=0.4)

    ax.set_xticks(x)
    ax.set_xticklabels([f'{m}\n({MODEL_DEPTH.get(m,"?")}) {MODEL_PARAMS.get(m,"?")}M'
                        for m in models_sorted], fontsize=9)
    ax.set_ylabel('Deletion AUC ↓  (meilleur = plus bas)', fontsize=11)
    ax.set_title('Figure 5 — Deletion AUC par méthode × profondeur du réseau (ImageNet)', fontsize=12)
    ax.legend(fontsize=9, ncol=2); ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(Path(PATHS['figures'])/'fig5_depth_effect.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Résultats ImageNet non disponibles — Figure 5 ignorée.')

## 11. Tableaux de résultats (1–8)

In [ ]:
# ─── Tableaux de résultats (1–8) ─────────────────────────────────────────────

def make_table(df, dataset, metric, model_names, method_names, title, ascending=True):
    """Génère un tableau méthode × modèle avec valeurs moyennes."""    sub = df[df.dataset == dataset]
    rows = []
    for mname in method_names:
        row = {'Méthode': METHOD_LABELS.get(mname, mname)}
        for model in model_names:
            val = sub[(sub.method==mname) & (sub.model==model)][metric].mean()
            row[model] = round(val, 4) if not np.isnan(val) else 'N/A'
        rows.append(row)
    tbl = pd.DataFrame(rows).set_index('Méthode')
    print(f'\n{"="*60}')
    print(f'{title}')
    print(f'{"="*60}')
    print(tbl.to_string())
    return tbl


if not results_df.empty:
    methods_main = ['gradcam','gradcam_pp','scorecam','layercam','ig_input','igcam_a','igcam_b']
    models_imagenet = [m for m in ['vgg16','resnet50','resnet152','densenet201'] if m in ACTIVE_MODELS]
    models_cub      = [m for m in ['resnet50','resnet152']                        if m in ACTIVE_MODELS]
    models_voc      = [m for m in ['vgg16','resnet50','resnet152']                if m in ACTIVE_MODELS]

    tables = {}

    if 'imagenet' in results_df['dataset'].values:
        tables['T1'] = make_table(results_df, 'imagenet', 'auc_deletion',  models_imagenet, methods_main,
                                   'Tableau 1 — Deletion AUC ↓  sur ImageNet')
        tables['T2'] = make_table(results_df, 'imagenet', 'auc_insertion', models_imagenet, methods_main,
                                   'Tableau 2 — Insertion AUC ↑  sur ImageNet', ascending=False)

    if 'voc' in results_df['dataset'].values:
        tables['T3'] = make_table(results_df, 'voc', 'pointing_game', models_voc, methods_main,
                                   'Tableau 3 — Pointing Game Accuracy ↑  sur PASCAL VOC 2007')

    if 'cub200' in results_df['dataset'].values:
        tables['T4'] = make_table(results_df, 'cub200', 'auc_deletion', models_cub, methods_main,
                                   'Tableau 4 — Deletion AUC ↓  sur CUB-200-2011 (fine-grained)')

    if 'isic' in results_df['dataset'].values:
        tbl5_methods = [m for m in methods_main if m in results_df['method'].values]
        tbl5 = results_df[results_df.dataset=='isic'].groupby('method')[
            ['auc_deletion','auc_insertion','energy_pointing','completeness_err_pct']].mean().round(4)
        tbl5.index = tbl5.index.map(lambda m: METHOD_LABELS.get(m, m))
        print('\n' + '='*60)
        print('Tableau 5 — ISIC 2019 avec ResNet-50')
        print('='*60)
        print(tbl5.to_string())
        tables['T5'] = tbl5

    # Tableau 6 : ablation N
    if not ablation_n_df.empty:
        print('\n' + '='*60)
        print('Tableau 6 — Ablation N (IG-CAM++ / ResNet-50 / ImageNet)')
        print('='*60)
        print(ablation_n_df.to_string(index=False))

    # Tableau 7 : ablation baseline
    if not abl_bl_df.empty:
        print('\n' + '='*60)
        print('Tableau 7 — Ablation baseline (IG-CAM++ / ResNet-50)')
        print('='*60)
        print(abl_bl_df.to_string(index=False))

    # Tableau 8 : A vs B (toutes métriques)
    if 'imagenet' in results_df['dataset'].values:
        t8 = results_df[(results_df.dataset=='imagenet') &
                        (results_df.method.isin(['igcam_a','igcam_b']))
                       ].groupby('method')[
            ['auc_insertion','auc_deletion','pointing_game','energy_pointing',
             'completeness_err_pct','time_ms']].mean().round(4)
        t8.index = t8.index.map(lambda m: METHOD_LABELS.get(m, m))
        print('\n' + '='*60)
        print('Tableau 8 — IG-CAM (A) vs IG-CAM++ (B) — synthèse')
        print('='*60)
        print(t8.to_string())
        tables['T8'] = t8

    # Sauvegarder
    for tname, tbl in tables.items():
        tbl.to_csv(Path(PATHS['results'])/f'{tname}.csv')
else:
    print('Pas de résultats — tableaux ignorés.')

## 11bis. ROAR — Remove And Retrain (optionnel)

In [ ]:
# ─── 11. ROAR — Remove And Retrain (optionnel, très coûteux) ────────────────
# Protocole : CUB-200 avec ResNet-50
# Coût estimé : ~168h GPU → exécuter sur cluster

ROAR_RATIOS  = [0.10, 0.30, 0.50, 0.70, 0.90]
ROAR_METHODS = ['gradcam', 'gradcam_pp', 'igcam_a', 'igcam_b']
RUN_ROAR     = False   # ← passer à True uniquement sur cluster GPU

def apply_roar_mask(image_tensor, cam, ratio):
    """Supprime les 'ratio' pixels les plus importants (remplace par 0)."""    img   = image_tensor.squeeze(0).cpu().numpy().copy()
    n_pix = img.shape[1] * img.shape[2]
    n_del = int(ratio * n_pix)
    order = np.argsort(cam.flatten())[::-1][:n_del]
    for c in range(3):
        flat = img[c].flatten(); flat[order] = 0; img[c] = flat.reshape(img[c].shape)
    return torch.tensor(img).unsqueeze(0).float()

if RUN_ROAR and 'cub200' in DATASETS:
    print('ROAR — CUB-200 (résultats à interpréter après ré-entraînement)')
    print('Note : ce bloc génère les images masquées. Le ré-entraînement')
    print('       doit être effectué séparément (voir scripts/train_roar.py).')
    roar_dir = Path(PATHS['results']) / 'roar_masked_images'
    roar_dir.mkdir(exist_ok=True)

    for ratio in ROAR_RATIOS:
        ratio_dir = roar_dir / f'ratio_{int(ratio*100):02d}'
        ratio_dir.mkdir(exist_ok=True)
        for mname in ROAR_METHODS:
            method_dir = ratio_dir / mname; method_dir.mkdir(exist_ok=True)
            for img_info in tqdm(DATASETS['cub200'][:200], desc=f'ROAR {mname} {ratio:.0%}'):
                cls, _ = predict_top1(MODELS['resnet50'], img_info['tensor'])
                cam, _, _ = EXPLAINERS['resnet50'].explain(mname, img_info['tensor'], cls)
                masked = apply_roar_mask(img_info['tensor'], cam, ratio)
                # Sauvegarder l'image masquée
                arr = masked.squeeze().permute(1,2,0).numpy()
                arr = np.clip((arr * np.array(STD) + np.array(MEAN)), 0, 1)
                Image.fromarray((arr*255).astype(np.uint8)).save(
                    method_dir / f'{img_info["image_id"]}.jpg')
    print(f'Images ROAR sauvegardées dans {roar_dir}')
    print('Lancer le ré-entraînement avec : python scripts/train_roar.py --roar_dir ' + str(roar_dir))
else:
    print('ROAR désactivé (RUN_ROAR=False) ou CUB-200 non disponible.')
    print('Activer en mettant RUN_ROAR=True et en ayant CUB-200 configuré.')

## 12. Conclusion et prochaines étapes

### Résultats clés attendus

**Hypothèse principale** : IG-CAM++ (B) surpasse Grad-CAM sur les réseaux profonds
(ResNet-152, DenseNet-201) car les Integrated Gradients compensent la saturation
des gradients sur les chemins d'activation longs.

**Hypothèse secondaire** : Le gain est plus marqué sur les tâches *fine-grained*
(CUB-200, ISIC) où les features discriminatives sont subtiles et localisées.

### Checklist avant soumission
- [ ] Résultats sur les 4 datasets × 5 modèles × 7 méthodes
- [ ] Tests de Wilcoxon significatifs (p < 0.05 / n_comparaisons)
- [ ] Ablation N confirmant N=50 comme compromis optimal
- [ ] Ablation baseline montrant la robustesse au choix du baseline
- [ ] ROAR sur CUB-200 (si ressources disponibles)
- [ ] Figures 1–5 finalisées
- [ ] Code publié sur GitHub avec seeds fixées
- [ ] Résultats loggés dans W&B / MLflow

### Conférences cibles
| Conférence | Deadline estimée |
|------------|-----------------|
| ECCV 2026 | Mars 2026 |
| CVPR 2027 | Novembre 2026 |
| AAAI 2027 | Août 2026 |
| ICCV 2027 | Mars 2027 |